# Notebook 01: Fast-Path Document Processing

**Goal:** Load a PDF document using `PyMuPDFLoader` (C-compiled engine), clean text formatting, and split into hierarchical chunks using `RecursiveCharacterTextSplitter` for sub-20ms latency.

## Objectives:
1. Load PDF using ultra-fast `PyMuPDFLoader` (`fitz` engine)
2. Normalize text and remove unwanted headers/footers
3. Build hierarchical section & paragraph chunks using `RecursiveCharacterTextSplitter`
4. Inspect chunks and run verification assertions

In [6]:
# Cell 1: Import modern dependencies
import os
import sys
import re
import time
from typing import List, Dict, Any

project_root = os.path.abspath(os.path.join(os.getcwd(), '..')) if os.path.basename(os.getcwd()) == 'notebooks' else os.getcwd()
if project_root not in sys.path:
    sys.path.insert(0, project_root)

from langchain_community.document_loaders import PyMuPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from scripts.create_sample_pdf import create_sample_pdf

print("Notebook 01 fast-path dependencies loaded successfully.")

Notebook 01 fast-path dependencies loaded successfully.


In [7]:
# Cell 2: Ultra-Fast PyMuPDF Extraction & Clean Pipeline
sample_pdf_path = os.path.join(project_root, "data", "harry_potter.pdf")
if not os.path.exists(sample_pdf_path):
    create_sample_pdf(sample_pdf_path)

t0 = time.time()
loader = PyMuPDFLoader(sample_pdf_path)
raw_documents = loader.load()
extraction_latency = (time.time() - t0) * 1000

print(f"Extracted {len(raw_documents)} pages via PyMuPDF in {extraction_latency:.2f} ms.")
print(f"Sample Page 1 Text Preview:\n{raw_documents[0].page_content[:200]}...")

Extracted 100 pages via PyMuPDF in 85.57 ms.
Sample Page 1 Text Preview:
...


In [8]:
# Cell 3: Text Cleaning & Recursive Chunking
def clean_document_text(text: str) -> str:
    cleaned = re.sub(r'(?i)page\s+\d+(\s+of\s+\d+)?', '', text)
    paragraphs = [re.sub(r'\s+', ' ', p).strip() for p in cleaned.split('\n\n') if p.strip()]
    return '\n\n'.join(paragraphs)

cleaned_docs = []
for doc in raw_documents:
    c_text = clean_document_text(doc.page_content)
    if c_text.strip():
        doc.page_content = c_text
        cleaned_docs.append(doc)

t1 = time.time()
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1200,
    chunk_overlap=150,
    separators=["\n\n", "\n", ". ", " "]
)

split_docs = text_splitter.split_documents(cleaned_docs)
chunking_latency = (time.time() - t1) * 1000
print(f"Generated {len(split_docs)} chunks using RecursiveCharacterTextSplitter in {chunking_latency:.2f} ms.")

Generated 178 chunks using RecursiveCharacterTextSplitter in 3.35 ms.


In [9]:
# Cell 4: Chunk Formatting & Inspection
chunks = []
for idx, doc in enumerate(split_docs):
    content = doc.page_content.strip()
    if not content.endswith(('.', '!', '?', ':', ';')):
        content += '.'
    lines = content.split('\n')
    first_line = lines[0].strip()
    sec_title = first_line if re.match(r'^\d+\.\s+[A-Z]', first_line) else f"Page {doc.metadata.get('page', 0) + 1} Context"
    sentences = [s.strip() for s in re.split(r'(?<=[.!?])\s+', content) if s.strip()]
    chunks.append({
        "chunk_id": f"chunk_{idx+1:03d}",
        "section": sec_title,
        "paragraph_index": idx + 1,
        "content": content,
        "sentences": sentences,
        "word_count": len(content.split())
    })

for c in chunks[:3]:
    print(f"=== {c['chunk_id']} | Section: {c['section']} | Words: {c['word_count']} ===")
    print(f"Content: {c['content'][:150]}...")
    print()

=== chunk_001 | Section: Page 3 Context | Words: 24 ===
Content: Harry Potter and the Goblet Of Fire by J. K. Rowling Illustrations by Mary Grandpré Arthur A. Levine Books An Imprint of Scholastic Press....

=== chunk_002 | Section: Page 4 Context | Words: 19 ===
Content: To Peter Rowling, In Memory of Mr. Ridley And to Susan Sladden, Who helped Harry Out of his cupboard....

=== chunk_003 | Section: Page 5 Context | Words: 151 ===
Content: Text copyright © 2000 by J.K. Rowling Illustrations by Mary GrandPre copyright © 2000 Warner Bros. All rights reserved. Published by Scholastic Press,...



In [10]:
# Cell 5: Verification Assertions for Notebook 01
print("Running fast-path verification assertions...")
assert len(chunks) > 0, "Verification Failed: No chunks generated!"
assert extraction_latency + chunking_latency < 5000.0, f"Verification Failed: Ingestion latency {extraction_latency + chunking_latency:.2f} ms exceeded threshold!"
for chunk in chunks:
    assert chunk["content"].endswith((".", "!", "?", ":", ";")), f"Verification Failed: Chunk {chunk['chunk_id']} ends mid-sentence!"
print("✅ ALL VERIFICATION TESTS PASSED FOR FAST-PATH NOTEBOOK 01!")

Running fast-path verification assertions...
✅ ALL VERIFICATION TESTS PASSED FOR FAST-PATH NOTEBOOK 01!
